# Generic Tuning Interface

In [1]:
from arbok_driver import ArbokDriver, Device, Measurement
from arbok_driver.examples.configurations import opx1000_config

2026-03-22 11:38:37,045 - qm - INFO     - Starting session: 1c943435-cb42-4211-b6f7-afdb53145e25


In [2]:
from arbok_driver.examples.configurations import (
    device_config, parity_init_conf, parity_read_conf
)

mock_device = Device(
    name = 'mock_device',
    opx_config = opx1000_config,
    divider_config = {},
    master_config = device_config
)

qm_driver = ArbokDriver('qm_driver', mock_device)

mock_measurement = Measurement(qm_driver, 'mock_measurement')

mock_measurement.add_subsequences_from_dict({
    'paraity_init_even': {'config': parity_init_conf},
    'parity_read_even': {'config': parity_read_conf},
    'parity_init_odd': {'config': parity_init_conf},
    'parity_read_odd': {'config': parity_read_conf}
})

In [3]:
from arbok_driver.generic_tunig_interface import CostStrategy

import numpy as np

class CustomStrategy(CostStrategy):
    def get_cost(self, results: dict) -> float:
        return float(np.random.rand())
    
parity_cost_strat = CustomStrategy(
    mock_measurement.available_gettables,
    [{mock_measurement.iteration: np.arange(100)}]
    )

In [4]:
[g.full_name for g in mock_measurement.available_gettables]

['qm_driver_mock_measurement_parity_read_even_ref__p1p2',
 'qm_driver_mock_measurement_parity_read_even_ref__p7p8',
 'qm_driver_mock_measurement_parity_read_even_read__p1p2',
 'qm_driver_mock_measurement_parity_read_even_read__p7p8',
 'qm_driver_mock_measurement_parity_read_even_diff__p1p2',
 'qm_driver_mock_measurement_parity_read_even_diff__p7p8',
 'qm_driver_mock_measurement_parity_read_even_state__p1p2',
 'qm_driver_mock_measurement_parity_read_even_state__p7p8',
 'qm_driver_mock_measurement_parity_read_odd_ref__p1p2',
 'qm_driver_mock_measurement_parity_read_odd_ref__p7p8',
 'qm_driver_mock_measurement_parity_read_odd_read__p1p2',
 'qm_driver_mock_measurement_parity_read_odd_read__p7p8',
 'qm_driver_mock_measurement_parity_read_odd_diff__p1p2',
 'qm_driver_mock_measurement_parity_read_odd_diff__p7p8',
 'qm_driver_mock_measurement_parity_read_odd_state__p1p2',
 'qm_driver_mock_measurement_parity_read_odd_state__p7p8']

In [5]:
mock_measurement.print_readable_snapshot()

qm_driver_mock_measurement:
	parameter       value
--------------------------------------------------------------------------------
f_larmor_Q1      :	27850000 (Hz)
f_larmor_Q2      :	30100000 (Hz)
f_larmor_Q3      :	26700000 (Hz)
f_larmor_Q4      :	48100000 (Hz)
f_larmor_Q5      :	45342000 (Hz)
f_larmor_Q6      :	50340000 (Hz)
f_larmor_Q7      :	45400000 (Hz)
f_larmor_Q8      :	42800000 (Hz)
f_larmor_qe1     :	1000000 (Hz)
gate_elements    :	['P1', 'J1', 'P2'] (N/A)
iteration        :	1 
qubit_elements   :	['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8'] (N/A)
readout_elements :	['SET1', 'SET2'] (N/A)
t_2pi_Q1         :	2500 (s)
t_2pi_Q2         :	2283 (s)
t_2pi_Q3         :	1000 (s)
t_2pi_Q4         :	1600 (s)
t_2pi_Q5         :	1642 (s)
t_2pi_Q6         :	1604 (s)
t_2pi_Q7         :	1136 (s)
t_2pi_Q8         :	1923 (s)
v_control_J1     :	-0.05 (V)
v_control_J2     :	-0 (V)
v_control_J3     :	-0.2 (V)
v_control_J4     :	0 (V)
v_control_J5     :	-0.19 (V)
v_control_J6     :	0 (V)
v_co

In [6]:
mock_measurement.initialize_tuning_interface(
    parameter_dicts = {
        't_ramp_init': {
            'qua_vars': {mock_measurement.parity_init_odd.arbok_params.t_ramp_over_crossing: 1},
            'bounds': (25, 1e4)
            },
        'read_detuning': {
            'qua_vars': {
                mock_measurement.parity_read_even.arbok_params.v_read['P1']: 1,
                mock_measurement.parity_read_even.arbok_params.v_read['P2']: -1,
                mock_measurement.parity_read_odd.arbok_params.v_read['P1']: 1,
                mock_measurement.parity_read_odd.arbok_params.v_read['P2']: -1,
                },
            'bounds': (-0.01, 0.01)
            },
        },
    cost_strategy = parity_cost_strat,
    verbose = True
)

Adding input stream for t_ramp_over_crossing (t_ramp_init)

Adding input stream for v_read_P1 (read_detuning)

Adding v_read_P2 to v_read_P1  input stream (factor: -1) (read_detuning)

Adding v_read_P1 to v_read_P1  input stream (factor: 1) (read_detuning)

Adding v_read_P2 to v_read_P1  input stream (factor: -1) (read_detuning)

Registered 16 gettables for measurement
Declared 1-dimensional parameter sweep of size 100 [100]


In [7]:
mock_measurement.print_qua_program_to_file('tet_parity.py')

In [8]:
mock_measurement.parity_read_even.arbok_params.v_read['P1'].qua

In [9]:
mock_measurement.parity_read_even.arbok_params.v_read['P2'].qua

In [10]:
mock_measurement.parity_read_even.arbok_params.v_read['P2'].call_method()